## NO TOOL CALL

In [ ]:
#&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
import json
import csv
import random
import re
from openai import OpenAI
import os
import time

# Initialize OpenAI client
client = OpenAI(api_key='')

# # Initialize NVIDIA API client
# client = OpenAI(
#     base_url="https://integrate.api.nvidia.com/v1",
#     api_key=""
# )
# MODEL = "meta/llama-3.1-405b-instruct"

# Load the JSON file
with open('D:\\Uni Study\\Sem-6\\7.My Datasets\\Func Schema\\3.functions_by_scenario_v3.json', 'r') as f:
    data = json.load(f)

# Function to generate a realistic user query using OpenAI API
def generate_query(function_info):
    system_prompt = f"""You are an Enterprise Digital Assistant that processes natural language queries across HR, CRM, Finance, Inventory, Sales, and IT systems to assist employees with business tools.

Context: You generate purely informational enterprise queries that request general guidance, explanations, or policies rather than specific actions. These queries should appear to be requesting legitimate business information but shouldn't trigger any backend function calls since they're seeking knowledge rather than system actions.

Constraints:
1. Each query must be STRICTLY 50-100 characters and written like a real business scenario.
2. Do not explicitly use the function names or parameter labels in the query. and use the function description for query generation.
3. Use realistic enterprise scenarios and business terminology in context, but adhere to the Prompt crafting Strategy.
4. For IDs (employee ID, case ID, request ID), take 1st ¾ letters followed by ¾ numbers for that ID. for instance - EMP456, CUST456, TICK003, PROD789, VEND001, TRACK123.
5. Use natural language, not code or pseudo-function calls. Questions should be diverse in style and tone. 

STRATEGY: HALLUCINATED FUNCTION - WRONG FUNCTION NAMES
 - Purpose: Test handling of non-existent functions.
 - Function Call Decision: NO
 - Details: Create queries that suggest plausible but non-existent functions. Use business terminology that sounds like a function but isn't in our schema. Make it similar to real functions but clearly different, ensuring no real function actually applies.

EXAMPLES:
- Instead of "schedule_performance_review": "Can you run the employee evaluation scheduler for EMP456 and generate the standard metrics dashboard?"  (Uses fake function-like terms: "employee evaluation scheduler" and "metrics dashboard")
- Instead of "process_invoice_payment": "Need to activate the vendor remittance protocol for the quarterly billing cycle with VEND234." (Uses fake function-like terms: "vendor remittance protocol" and "quarterly billing cycle")
"""

    user_prompt = f"""Generate a Language Diversity and Ambiguity user query:

Function: {function_info['function_name']}
Description: {function_info['function_description']}
Required Parameters: {function_info['parameters'].get('required', [])}
All Parameters: {json.dumps(function_info['parameters'].get('properties', {}), indent=2)}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-2025-04-14",
            # model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.4  # Balanced for variety while maintaining structure
        )
        
        query = response.choices[0].message.content.strip()

        # Validate query length (50-150 characters)
        if len(query) < 50 or len(query) > 150:
            print(f"Warning: Query length {len(query)} is outside 50-150 character range")

        return query
    except Exception as e:
        print(f"Error generating query: {e}")
        return None

# Extract all functions from the JSON
functions = []
for category, func_list in data.items():
    for func in func_list:
        if func['type'] == 'function':
            functions.append({
                'category': category,
                'function_name': func['function']['name'],
                'function_description': func['function']['description'],
                'parameters': func['function']['parameters']
            })

# Create CSV file for query generation only
with open('D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\enterprise_queries_only.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['category', 'function_name', 'user_query', 'query_length', 'retry_count']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for i, func in enumerate(functions):
        print(f"Processing function {i+1}/{len(functions)}: {func['function_name']}")
        
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            # Generate user query
            user_query = generate_query(func)
            
            if user_query:
                print(f"Generated query (attempt {retry_count + 1}): {user_query}")
                
                # Check query length constraints
                if 50 <= len(user_query) <= 150:
                    success = True
                    print(f"Query length: {len(user_query)} characters - Valid!")
                else:
                    print(f"Query length: {len(user_query)} characters - Retrying...")
                    retry_count += 1
                    continue
                
                # Write successful query to CSV
                writer.writerow({
                    'category': func['category'],
                    'function_name': func['function_name'],
                    'user_query': user_query,
                    'query_length': len(user_query),
                    'retry_count': retry_count
                })
                
                print(f"Success after {retry_count + 1} attempts")
                print("-" * 50)
                
            else:
                print(f"Failed to generate query, retrying...")
                retry_count += 1
        
        if not success:
            print(f"Failed to generate valid query for {func['function_name']} after {max_retries} attempts")
            # Still write the last attempt to CSV for analysis
            writer.writerow({
                'category': func['category'],
                'function_name': func['function_name'],
                'user_query': user_query or "Failed to generate",
                'query_length': len(user_query) if user_query else 0,
                'retry_count': max_retries
            })

print("Query generation completed. Results saved to enterprise_queries_only.csv")

## TOOL CALL - 1 Date and Time Expression 

In [ ]:
##############################################
import json
import csv
import random
import re
from openai import OpenAI
import os
import time

# Initialize OpenAI client
client = OpenAI(api_key='')

# # Initialize NVIDIA API client
# client = OpenAI(
#     base_url="https://integrate.api.nvidia.com/v1",
#     api_key=""
# )
# MODEL = "meta/llama-3.1-405b-instruct"

# Load the JSON file
with open('D:\\Uni Study\\Sem-6\\7.My Datasets\\Func Schema\\3.functions_by_scenario_v3.json', 'r') as f:
    data = json.load(f)

# List of functions for date and time expression variations
date_time_functions = [
    "schedule_performance_review",
    "process_leave_request",
    "generate_payroll_report",
    "manage_customer_campaign",
    "conduct_inventory_audit",
    "generate_financial_statements",
    "execute_month_end_close",
    "initiate_remote_session",
    "request_network_access",
    "deploy_software",
    "create_purchase_order",
    "request_vendor_quotation",
    "conduct_spend_analysis",
    "generate_sales_report",
    "generate_sales_forecast",
    "generate_contract_renewal_reminders",
    "process_sales_commission",
    "conduct_compliance_audit",
    "generate_compliance_report",
    "book_meeting_room",
    "monitor_energy_consumption",
    "update_ticket_status",
    "create_performance_review",
    "schedule_performance_review_meeting",
    "request_peer_feedback",
    "schedule_meeting",
    "reschedule_meeting",
    "cancel_meeting",
    "create_campaign",
    "analyze_campaign_performance",
    "track_ad_performance",
    "schedule_interview",
    "send_offer_letter"
]

# Function to generate a realistic user query using OpenAI API
def generate_query(function_info):
    system_prompt = f"""You are an Enterprise Digital Assistant that processes natural language queries across HR, CRM, Finance, Inventory, Sales, and IT systems to assist employees with business tools.

Context: You generate user queries that implicitly trigger backend enterprise functions across various domains. Each function has a specific schema, and parameters. Queries must reflect realistic, diverse user needs in a business setting.

Constraints:
1. Each query must be STRICTLY 50-150 characters and written like a real business scenario.
2. Do not explicitly use the function names or parameter labels in the query. Use the function description for query generation. 
3. Use realistic enterprise scenarios and business terminology in context, but adhere to the Prompt crafting Strategy.
4. For IDs (employee ID, case ID, request ID), use 1st ¾ letters followed by ¾ numbers for that ID. For instance - EMP456, CUST456, TICK003, PROD789, VEND001, TRACK123.
5. Use natural language, not code or pseudo-function calls. Questions should be diverse in style and tone.

STRATEGY: DATE AND TIME EXPRESSION VARIATIONS
   - Purpose: Test model's ability to interpret and normalize diverse date/time formats into function parameters.
   - Function Call Decision: YES
   - Details: Create queries for functions with date/time parameters using various formats and styles. Include:
     * Different date formats (US: MM/DD/YYYY, UK: DD/MM/YYYY, ISO: YYYY-MM-DD)
     * Different time formats (12h vs 24h)
     * Cultural variations ("quarter past", "half ten", etc.)
     * Never use ambiguous expressions like "next Monday", "two weeks from now", "today", "tomorrow", "end of month", 
       or "before the holidays" as these don't provide stable ground truth.
     * Always use specific calendar dates and clock times that can be normalized unambiguously.

EXAMPLES:
- Instead of "schedule_performance_review": "Need to set up a review with EMP456 on 15/03/2025 at quarter past two in the afternoon."
- Instead of "create_meeting": "Reserve the conference room for our team on 05/12/2024 from 14:30 to 16:00 hours."
- Instead of "process_invoice_payment": "Please pay vendor VEND123's invoice by September 30th, it's due at half past three PM."
"""

    user_prompt = f"""Generate a Language Diversity and Ambiguity user query:

Function: {function_info['function_name']}
Description: {function_info['function_description']}
Required Parameters: {function_info['parameters'].get('required', [])}
All Parameters: {json.dumps(function_info['parameters'].get('properties', {}), indent=2)}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-2025-04-14",
            # model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.4  # Balanced for variety while maintaining structure
        )
        
        query = response.choices[0].message.content.strip()

        # Validate query length (50-150 characters)
        if len(query) < 50 or len(query) > 150:
            print(f"Warning: Query length {len(query)} is outside 50-150 character range")

        return query
    except Exception as e:
        print(f"Error generating query: {e}")
        return None

def get_actual_function_call(user_query, tools):
    """Get actual function call from model given user query and tools"""
    try:
        start_time = time.time()
        
        response = client.chat.completions.create(
            model="gpt-4.1-2025-04-14",
            messages=[{"role": "user", "content": user_query}],
            tools=tools, # Pass only the specific tool for this function
            tool_choice="auto",
            temperature=0.0,
            parallel_tool_calls=False
        )
        
        execution_time = round(time.time() - start_time, 2)
        
        # Extract token usage
        usage = response.usage
        token_usage = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "execution_time": execution_time
        }
        
        # Extract tool calls
        tool_calls = response.choices[0].message.tool_calls
        if tool_calls:
            function_calls = [
                {
                    "name": tc.function.name,
                    "arguments": json.loads(tc.function.arguments)
                } for tc in tool_calls
            ]
            return json.dumps(function_calls), token_usage
        else:
            return "None (No function calls)", token_usage
            
    except Exception as e:
        return f"Error: {str(e)}", {"error": str(e)}

# Extract only the specified date_time_functions from the JSON
functions = []
for category, func_list in data.items():
    for func in func_list:
        if func['type'] == 'function' and func['function']['name'] in date_time_functions:
            functions.append({
                'category': category,
                'function_name': func['function']['name'],
                'function_description': func['function']['description'],
                'parameters': func['function']['parameters']
            })

# Continue with the rest of your existing code...
with open('D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\enterprise_queries_with_actual_calls.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['category', 'function_name', 'user_query', 'actual_function_call', 'token_usage', 'execution_time', 'retry_count']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for i, func in enumerate(functions):
        print(f"Processing function {i+1}/{len(functions)}: {func['function_name']}")
        
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            # Generate user query
            user_query = generate_query(func)
            
            if user_query:
                print(f"Generated query (attempt {retry_count + 1}): {user_query[:100]}...")
                
                # Create single tool for this specific function to ensure targeted calling
                single_tool = None
                for category, func_list in data.items():
                    for tool_func in func_list:
                        if tool_func['type'] == 'function' and tool_func['function']['name'] == func['function_name']:
                            single_tool = [tool_func]  # Only pass the target tool
                            break
                    if single_tool:
                        break
                
                # Get actual function call from model with only the target tool
                actual_call, usage_info = get_actual_function_call(user_query, single_tool)
                
                # Check if we got a successful function call
                if actual_call != "None (No function calls)" and not actual_call.startswith("Error:"):
                    try:
                        # Parse the function call to check if it's valid
                        parsed_call = json.loads(actual_call)
                        if isinstance(parsed_call, list) and len(parsed_call) == 1:
                            # Exactly one function call - success!
                            success = True
                        elif isinstance(parsed_call, list) and len(parsed_call) > 1:
                            print(f"Multiple function calls detected, retrying...")
                            retry_count += 1
                            continue
                    except json.JSONDecodeError:
                        print(f"Invalid JSON in function call, retrying...")
                        retry_count += 1
                        continue
                else:
                    print(f"No function call generated, retrying...")
                    retry_count += 1
                    continue
                
                # Format token usage
                if "error" not in usage_info:
                    token_info = f"Prompt: {usage_info['prompt_tokens']}, Completion: {usage_info['completion_tokens']}, Total: {usage_info['total_tokens']}"
                    exec_time = f"{usage_info['execution_time']} seconds"
                else:
                    token_info = f"Error: {usage_info['error']}"
                    exec_time = "Error"
                
                writer.writerow({
                    'category': func['category'],
                    'function_name': func['function_name'],
                    'user_query': user_query,
                    'actual_function_call': actual_call,
                    'token_usage': token_info,
                    'execution_time': exec_time,
                    'retry_count': retry_count
                })
                
                print(f"Actual function call: {actual_call[:100] if len(str(actual_call)) > 100 else actual_call}")
                print(f"Token usage: {token_info}")
                print(f"Success after {retry_count + 1} attempts")
                print("-" * 50)
                
            else:
                print(f"Failed to generate query, retrying...")
                retry_count += 1
        
        if not success:
            print(f"Failed to generate successful query for {func['function_name']} after {max_retries} attempts")
            # Still write the last attempt to CSV for analysis
            writer.writerow({
                'category': func['category'],
                'function_name': func['function_name'],
                'user_query': user_query or "Failed to generate",
                'actual_function_call': actual_call if 'actual_call' in locals() else "Failed",
                'token_usage': "Failed",
                'execution_time': "Failed",
                'retry_count': max_retries
            })

print("Query generation and function calling completed. Results saved to enterprise_queries_with_actual_calls.csv")

## Tool Call - 2

In [ ]:
##############################################
import json
import csv
import random
import re
from openai import OpenAI
import os
import time

# Initialize OpenAI client
client = OpenAI(api_key='')

# # Initialize NVIDIA API client
# client = OpenAI(
#     base_url="https://integrate.api.nvidia.com/v1",
#     api_key=""
# )
# MODEL = "meta/llama-3.1-405b-instruct"

# Load the JSON file
with open('D:\\Uni Study\\Sem-6\\7.My Datasets\\Func Schema\\3.functions_by_scenario_v3.json', 'r') as f:
    data = json.load(f)

# Function to generate a realistic user query using OpenAI API
def generate_query(function_info):
    system_prompt = f"""You are an Enterprise Digital Assistant that processes natural language queries across HR, CRM, Finance, Inventory, Sales, and IT systems to assist employees with business tools.

Context: You generate user queries that implicitly trigger backend enterprise functions across various domains. Each function has a specific schema, and parameters.

Constraints:
1. Each query must be STRICTLY 50-100 characters and written like a real business scenario.
2. Do not explicitly use the function names or parameter labels in the query. and use the function description for query generation.
3. Use realistic enterprise scenarios and business terminology in context, but adhere to the Prompt crafting Strategy.
4. For IDs (employee ID, case ID, request ID), take 1st ¾ letters followed by ¾ numbers for that ID. for instance - EMP456, CUST456, TICK003, PROD789, VEND001, TRACK123.
5. Use natural language, not code or pseudo-function calls. Questions should be diverse in style and tone. 

STRATEGY: TOOL SELECTION AMBIGUITY
   - Purpose: Multiple functions could potentially handle the request; test selection of the most appropriate function.
 - Function Call Decision: YES
 - Details: Use vague but actionable prompts where multiple functions might apply (e.g., "Plan my team’s schedule" could map to calendar or resource allocation functions). Rephrase terms to avoid direct function names while ensuring the query aligns with the target function's intent and includes all required parameters.
EXAMPLES:
- Instead of "create_performance_review": "Time to do the quarterly check-in for EMP789. Can you set it up with their manager EMP101?"  (Could map to: create_performance_review, schedule_performance_review_meeting, or request_peer_feedback)
- Instead of "process_invoice_payment": "Got that pending bill from VEND789 for $5000 - needs to be handled before Friday"(Could map to: process_invoice_payment, handle_expense_reimbursement, or create_purchase_order)



"""
    user_prompt = f"""Generate tool selection AMBIGUITY user query for the following enterprise function:

Function: {function_info['function_name']}
Description: {function_info['function_description']}
Required Parameters: {function_info['parameters'].get('required', [])}
All Parameters: {json.dumps(function_info['parameters'].get('properties', {}), indent=2)}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-2024-11-06",
            # model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.4  # Balanced for variety while maintaining structure
        )
        
        query = response.choices[0].message.content.strip()

        # Validate query length (50-100 characters)
        if len(query) < 50 or len(query) > 100:
            print(f"Warning: Query length {len(query)} is outside 50-100 character range")

        return query
    except Exception as e:
        print(f"Error generating query: {e}")
        return None

def get_actual_function_call(user_query, tools):
    """Get actual function call from model given user query and tools"""
    try:
        start_time = time.time()
        
        response = client.chat.completions.create(
            model="gpt-4o-2024-08-06",
            messages=[{"role": "user", "content": user_query}],
            tools=tools, # Pass only the specific tool for this function
            tool_choice="auto",
            temperature=0.0,
            parallel_tool_calls=False
        )
        
        execution_time = round(time.time() - start_time, 2)
        
        # Extract token usage
        usage = response.usage
        token_usage = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "execution_time": execution_time
        }
        
        # Extract tool calls
        tool_calls = response.choices[0].message.tool_calls
        if tool_calls:
            function_calls = [
                {
                    "name": tc.function.name,
                    "arguments": json.loads(tc.function.arguments)
                } for tc in tool_calls
            ]
            return json.dumps(function_calls), token_usage
        else:
            return "None (No function calls)", token_usage
            
    except Exception as e:
        return f"Error: {str(e)}", {"error": str(e)}

# Extract all functions from the JSON
functions = []
for category, func_list in data.items():
    for func in func_list:
        if func['type'] == 'function':
            functions.append({
                'category': category,
                'function_name': func['function']['name'],
                'function_description': func['function']['description'],
                'parameters': func['function']['parameters']
            })

with open('D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\enterprise_queries_with_actual_calls.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['category', 'function_name', 'user_query', 'actual_function_call', 'token_usage', 'execution_time', 'retry_count']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for i, func in enumerate(functions):
        print(f"Processing function {i+1}/{len(functions)}: {func['function_name']}")
        
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            # Generate user query
            user_query = generate_query(func)
            
            if user_query:
                print(f"Generated query (attempt {retry_count + 1}): {user_query[:100]}...")
                
                # Create single tool for this specific function to ensure targeted calling
                single_tool = None
                for category, func_list in data.items():
                    for tool_func in func_list:
                        if tool_func['type'] == 'function' and tool_func['function']['name'] == func['function_name']:
                            single_tool = [tool_func]  # Only pass the target tool
                            break
                    if single_tool:
                        break
                
                # Get actual function call from model with only the target tool
                actual_call, usage_info = get_actual_function_call(user_query, single_tool)
                
                # Check if we got a successful function call
                if actual_call != "None (No function calls)" and not actual_call.startswith("Error:"):
                    try:
                        # Parse the function call to check if it's valid
                        parsed_call = json.loads(actual_call)
                        if isinstance(parsed_call, list) and len(parsed_call) == 1:
                            # Exactly one function call - success!
                            success = True
                        elif isinstance(parsed_call, list) and len(parsed_call) > 1:
                            print(f"Multiple function calls detected, retrying...")
                            retry_count += 1
                            continue
                    except json.JSONDecodeError:
                        print(f"Invalid JSON in function call, retrying...")
                        retry_count += 1
                        continue
                else:
                    print(f"No function call generated, retrying...")
                    retry_count += 1
                    continue
                
                # Format token usage
                if "error" not in usage_info:
                    token_info = f"Prompt: {usage_info['prompt_tokens']}, Completion: {usage_info['completion_tokens']}, Total: {usage_info['total_tokens']}"
                    exec_time = f"{usage_info['execution_time']} seconds"
                else:
                    token_info = f"Error: {usage_info['error']}"
                    exec_time = "Error"
                
                writer.writerow({
                    'category': func['category'],
                    'function_name': func['function_name'],
                    'user_query': user_query,
                    'actual_function_call': actual_call,
                    'token_usage': token_info,
                    'execution_time': exec_time,
                    'retry_count': retry_count
                })
                
                print(f"Actual function call: {actual_call[:100] if len(str(actual_call)) > 100 else actual_call}")
                print(f"Token usage: {token_info}")
                print(f"Success after {retry_count + 1} attempts")
                print("-" * 50)
                
            else:
                print(f"Failed to generate query, retrying...")
                retry_count += 1
        
        if not success:
            print(f"Failed to generate successful query for {func['function_name']} after {max_retries} attempts")
            # Still write the last attempt to CSV for analysis
            writer.writerow({
                'category': func['category'],
                'function_name': func['function_name'],
                'user_query': user_query or "Failed to generate",
                'actual_function_call': actual_call if 'actual_call' in locals() else "Failed",
                'token_usage': "Failed",
                'execution_time': "Failed",
                'retry_count': max_retries
            })

print("Query generation and function calling completed. Results saved to enterprise_queries_with_actual_calls.csv")

Processing function 1/45: get_employee_profile
Generated query (attempt 1): Can you pull up everything we have on EMP123? Need to review their past contributions and roles....
Actual function call: [{"name": "get_employee_profile", "arguments": {"employee_id": "EMP123"}}]
Token usage: Prompt: 82, Completion: 17, Total: 99
Success after 1 attempts
--------------------------------------------------
Processing function 2/45: schedule_performance_review
Generated query (attempt 1): Let's arrange a check-in for EMP123 with their supervisor EMP456. It's time for that annual discussi...
No function call generated, retrying...
Generated query (attempt 2): Let's arrange a session for EMP123 with their supervisor EMP456 for the upcoming quarterly review....
No function call generated, retrying...
Generated query (attempt 3): Let's arrange a session for EMP123 with their supervisor EMP456. It's time for a quarterly chat....
No function call generated, retrying...
Failed to generate successful que

## Query via meta/llama-3.1-405b-instruct

In [ ]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = ""
)

completion = client.chat.completions.create(
  model="meta/llama-3.1-405b-instruct",
  messages=[{"role":"user","content":"Hello, who won the world series in 2020?"}],
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

The final answer is 5.